In [ ]:
%run ../globalvariables

In [ ]:
%run ../lakehousefunction

In [ ]:
NOTEBOOK = "silver/aire"

In [ ]:
# Unpivot wide days to long
errors = []
success = False

try:
    spark.table(f"{BRONZE_TABLE}.aire").createOrReplaceTempView("bronze_aire")

    magnitud_case = (
        "CASE magnitud "
        + " ".join(f"WHEN {c} THEN '{l}'" for c, l in MAGNITUD_LABELS.items())
        + " ELSE CAST(magnitud AS STRING) END"
    )

    days = [
        f"""
        SELECT CAST(estacion AS INT) AS estacion, {magnitud_case} AS magnitud,
               make_date(CAST(ano AS INT), CAST(mes AS INT), {d}) AS fecha,
               CAST(d{d:02d} AS DOUBLE) AS dato, v{d:02d} AS validez
        FROM bronze_aire
        WHERE {d} <= dayofmonth(
            last_day(make_date(CAST(ano AS INT), CAST(mes AS INT), 1))
        )
        """
        for d in range(1, 32)
    ]

    aire = spark.sql(f"SELECT DISTINCT * FROM ({' UNION ALL '.join(days)})")

    if not write_silver(aire, "aire"):
        raise Exception("write_silver returned False")
    success = True
    print(f"aire: {aire.count()} rows")
except Exception as e:
    errors.append(error_record(NOTEBOOK, e))
    print(f"fail aire: {type(e).__name__}: {e}")

In [ ]:
print(f"aire: {'SUCCESS' if success else 'FAILED'}")
log_errors(errors)